In [1]:
import sys, os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)
print('Project root:', project_root)

Project root: /Users/somita/Hospital_Supply_Chain_Bot


In [2]:
from src.config import check_ollama_running
from src.retriever import load_retriever
from src.chain import load_llm, run_test_queries
from src.database import (
    get_kpi_counts, get_critical_items, get_restock_alerts,
    get_vendor_risks, get_spend_summary, get_staff_summary,
    get_overstock_items, get_patient_diagnosis_summary
)
running, msg = check_ollama_running()
print('Ollama:', msg)
if not running:
    raise RuntimeError('Start Ollama: brew services start ollama')

Ollama: Llama 3 · Ollama · Live


In [3]:
_, _, retriever = load_retriever()
llm = load_llm()
print('Pipeline ready')

2026-04-12 18:44:19 | INFO | inventra.retriever | Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
/Users/somita/Hospital_Supply_Chain_Bot/src/retriever.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
2026-04-12 18:44:20 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 18:44:20 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-04-12 18:44:21 | INFO | httpx | HTTP

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-12 18:44:21 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-04-12 18:44:21 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-04-12 18:44:21 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-04-12 18:44:21 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/prepro

Pipeline ready


In [4]:

print('KPIs:'); print(get_kpi_counts())
print('\nCritical:'); print(get_critical_items())
print('\nVendors:');  print(get_vendor_risks())
print('\nSpend:');    print(get_spend_summary())
print('\nStaff:');    print(get_staff_summary())
print('\nOverstock:');print(get_overstock_items())

KPIs:
{'critical': 298, 'restock': 299, 'delayed': 5, 'total_spend_millions': 12.4}

Critical:
       Item_Name  Current_Stock  Days_Until_Stockout  Restock_Lead_Time  \
0  Surgical Mask             69                  0.2                 14   
1  X-Ray Machine            139                  0.3                 19   
2         Gloves            156                  0.3                 24   
3  X-Ray Machine             93                  0.3                  2   
4  Surgical Mask            160                  0.3                 17   

   Restock_Urgency_Days Vendor_ID  
0                 -13.8      V001  
1                 -18.7      V003  
2                 -23.7      V001  
3                  -1.7      V003  
4                 -16.7      V001  

Vendors:
        Vendor_Name   Item_Supplied  Avg_Lead_Time_days  \
0      EquipMed Co.      Ventilator                  30   
1      EquipMed Co.         Iv Drip                  30   
2      EquipMed Co.  Surgical Table                

In [5]:
run_test_queries(llm, retriever)

2026-04-12 18:45:00 | INFO | inventra.chain | Processing: Which items are most critical right now?



Q1: Which items are most critical right now?


2026-04-12 18:46:08 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"
2026-04-12 18:46:27 | INFO | inventra.chain | Response generated (628 chars)
2026-04-12 18:46:27 | INFO | inventra.chain | Processing: Is any vendor running late and what should I do?


Summary: Critical stockouts detected.

Top issues:
1. Surgical Mask — 69 units, 0.2 days until stockout, lead time 14d, vendor V001
2. X-Ray Machine — 139 units, 0.3 days until stockout, lead time 19d, vendor V003
3. Gloves — 156 units, 0.3 days until stockout, lead time 24d, vendor V001

Actions:
1. Restock Surgical Mask with 69 units as soon as possible.
2. Expedite X-Ray Machine order to avoid stockout.
3. Monitor Gloves inventory closely and restock if necessary.

Please note that these items are most critical due to their low stock levels and short lead times, which may result in stockouts if not addressed promptly.

Q2: Is any vendor running late and what should I do?


2026-04-12 18:46:33 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"
2026-04-12 18:46:45 | INFO | inventra.chain | Response generated (681 chars)
2026-04-12 18:46:45 | INFO | inventra.chain | Processing: What were our highest supply expenses?


...

Summary: Top expenses detected.

Key findings:
1. Supplies: $4,906,191 total, 188 transactions, avg $26,097
2. Staffing: $3,842,109 total, 167 transactions, avg $23,007
3. Equipment: $3,611,946 total, 145 transactions, avg $24,910

Actions:
1. Review and optimize supply chain processes to reduce unnecessary expenses.
2. Analyze transaction data to identify areas for cost savings.
3. Consider implementing cost-saving measures such as bulk purchasing or negotiating with vendors.

Note: The top three expense categories are Supplies, Staffing, and Equipment, which account for approximately 75% of the total expenses.

Q4: What supplies do ICU patients use most?


2026-04-12 18:47:13 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"
2026-04-12 18:47:23 | INFO | inventra.chain | Response generated (581 chars)
2026-04-12 18:47:23 | INFO | inventra.chain | Processing: Which staff departments have the highest overtime?


Summary: Top supplies used by ICU patients detected.

Top supplies:
1. Gloves — 76 units, 0.3 days left, lead time 24d, vendor V001
2. Iv Drip — 62 patients, avg 9.6 bed days
3. Surgical Mask — 61 patients, avg 10.4 bed days

Actions:
1. Expedite orders for Gloves and Iv Drips to ensure timely delivery.
2. Consider sourcing alternatives or negotiating with other vendors to mitigate delays.
3. Monitor vendor performance closely and adjust strategies as needed.

Note: ICU patients use a combination of consumable supplies (Gloves, Iv Drip) and medical equipment (Surgical Mask).

Q5: Which staff departments have the highest overtime?


2026-04-12 18:47:30 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"
2026-04-12 18:47:40 | INFO | inventra.chain | Response generated (646 chars)


Summary: Top staff departments with high overtime detected.

Key findings:
1. Nurse / ER: 61 shifts, avg 9.5h worked, avg 2.1h overtime
2. Technician / General Ward: 58 shifts, avg 9.5h worked, avg 2.1h overtime
3. Surgeon / ER: 52 shifts, avg 9.6h worked, avg 1.9h overtime

Actions:
1. Review staffing schedules to identify areas for optimization and reduce overtime.
2. Consider hiring additional staff or adjusting shift assignments to alleviate workload.
3. Monitor overtime trends closely and adjust strategies as needed.

Note: The top three staff departments with high overtime are Nurse / ER, Technician / General Ward, and Surgeon / ER.
